# Project Phase 1: Data Extraction & Pipeline Architecture
## Objective: 
Создание репрезентативной, очищенной и химически валидированной выборки металлоорганических соединений с известной противораковой активностью (pIC50) против рака яичников на основе базы данных ChEMBL (v37).

Извлечение металлокомплексов из реляционных баз данных — нестандартная задача. Большинство хемоинформатических пакетов (включая RDKit) ориентированы на классическую органическую химию (ковалентные связи). Координационные, дативные и многоцентровые связи часто вызывают нарушения валентности при стандартном парсинге SMILES. Данный этап демонстрирует построение отказоустойчивого пайплайна экстракции и in silico реконструкции координационных сфер.

In [2]:
import sqlite3
import pandas as pd
import re

# 1. ПОДКЛЮЧЕНИЕ К БАЗЕ
db_path = "chembl_37/chembl_37_sqlite/chembl_37.db"
conn = sqlite3.connect(db_path)

print("Выгружаем сырые данные по металлам вместе с биологическими мишенями...")

# Наш список металлов
metal_symbols = ["Pt", "Ru", "Ir", "Au", "Pd", "Os", "Rh", "Re", "Fe", "Co", "Ni"]
metal_conditions = " OR ".join([f"prop.full_molformula LIKE '%{m}%'" for m in metal_symbols])

# SQL-запрос, который вытаскивает id, формулу, pIC50 и НАЗВАНИЕ ТЕСТА/КЛЕТОК (pref_name)
query = f"""
SELECT 
    m.chembl_id,
    prop.full_molformula AS chemical_formula,
    act.pchembl_value AS pIC50,
    t.pref_name AS target_name
FROM activities act
JOIN molecule_dictionary m ON act.molregno = m.molregno
JOIN compound_properties prop ON m.molregno = prop.molregno
JOIN assays a ON act.assay_id = a.assay_id
JOIN target_dictionary t ON a.tid = t.tid
WHERE act.pchembl_value IS NOT NULL
  AND ({metal_conditions})
"""

df_raw = pd.read_sql_query(query, conn)
conn.close()

print(f"Выгружено сырых записей: {len(df_raw)}")

# 2. ПЕРВИЧНАЯ ФИЛЬТРАЦИЯ ИСТИННЫХ МЕТАЛЛОВ И УГЛЕРОДА (C >= 6)
true_metals = set(metal_symbols)

def clean_and_check(formula):
    if not isinstance(formula, str): return False
    # Валидация элементов
    tokens = set(re.findall(r'[A-Z][a-z]?', formula))
    if tokens.isdisjoint(true_metals): return False
    
    # Считаем углерод
    clean_formula = re.sub(r'[\+\-]\d*', '', formula)
    matches = re.findall(r'([A-Z][a-z]*)(\d*)', clean_formula)
    for element, count in matches:
        if element == 'C' and (int(count) if count else 1) >= 6:
            return True
    return False

df_cleaned = df_raw[df_raw["chemical_formula"].apply(clean_and_check)].copy()
print(f"Осталось истинной металлоорганики (C >= 6): {len(df_cleaned)}")

print("\n=== ТОП-20 БИОЛОГИЧЕСКИХ МИШЕНЕЙ (КЛЕТОЧНЫХ ЛИНИЙ) ===")
print(df_cleaned['target_name'].value_counts().head(20))

Выгружаем сырые данные по металлам вместе с биологическими мишенями...
Выгружено сырых записей: 219819
Осталось истинной металлоорганики (C >= 6): 2219

=== ТОП-20 БИОЛОГИЧЕСКИХ МИШЕНЕЙ (КЛЕТОЧНЫХ ЛИНИЙ) ===
target_name
A2780                                     223
CH1                                       188
SK-OV-3                                   113
B16                                        98
NON-PROTEIN TARGET                         96
L1210                                      76
Plasmodium falciparum                      73
MCF7                                       62
41M                                        59
Sodium-dependent dopamine transporter      42
Calu-6                                     39
Estrogen receptor                          38
P388                                       37
SK-MEL-25                                  34
Sodium-dependent serotonin transporter     33
LXF-289 cell line                          30
HX62 cell line                             3

Из полученной статистики для клеточных линий было решено взять данные для 4 линий рака яичников, имеющих наибольшее число строк: 'A2780', 'CH1', 'SK-OV-3', '41M'

In [3]:
import pandas as pd

# Списки линий, относящихся к раку яичников в нашем ТОПе
ovarian_lines = ['A2780', 'CH1', 'SK-OV-3', '41M']

print("1. Фильтрация данных по раку яичников...")
df_ovarian = df_cleaned[df_cleaned['target_name'].isin(ovarian_lines)].copy()
print(f"-> Найдено {len(df_ovarian)} записей активностей для выбранных линий.")

# 2. АГРЕГАЦИЯ ДУБЛИКАТОВ С УЧЕТОМ БИОЛОГИИ
# Берем медиану pIC50 для конкретного соединения внутри КОНКРЕТНОЙ линии клеток
print("2. Агрегация дубликатов по связке Молекула + Клеточная линия...")
df_grouped = df_ovarian.groupby(['chembl_id', 'chemical_formula', 'target_name'], as_index=False)['pIC50'].median()
print(f"-> Получено {len(df_grouped)} чистых, уникальных биологических точек для ML.")

# 3. ПОДГОТОВКА СПИСКА ДЛЯ ИИ-ПЕРЕВОДА
# Давай сразу выгрузим из ChEMBL исходные SMILES, чтобы понять, сколько у нас пропусков
import sqlite3
db_path = "chembl_37/chembl_37_sqlite/chembl_37.db"
conn = sqlite3.connect(db_path)

query_structures = """
SELECT m.chembl_id, str.canonical_smiles
FROM molecule_dictionary m
LEFT JOIN compound_structures str ON m.molregno = str.molregno
"""
df_str = pd.read_sql_query(query_structures, conn)
conn.close()

# Объединяем наши биологические точки с базовыми SMILES из ChEMBL
df_merged = pd.merge(df_grouped, df_str, on='chembl_id', how='left')

nan_smiles_count = df_merged['canonical_smiles'].isna().sum()
print(f"\nИз {len(df_merged)} уникальных точек: ")
print(f"✅ SMILES уже есть в ChEMBL: {len(df_merged) - nan_smiles_count}")
print(f"⚠️ SMILES отсутствует (нужно восстановить через ИИ): {nan_smiles_count}")

# Сохраняем промежуточную чистую биологическую матрицу
df_merged.to_csv("ovarian_cancer_base.csv", index=False)
print("\nБазовая матрица сохранена в 'ovarian_cancer_base.csv'")

1. Фильтрация данных по раку яичников...
-> Найдено 583 записей активностей для выбранных линий.
2. Агрегация дубликатов по связке Молекула + Клеточная линия...
-> Получено 316 чистых, уникальных биологических точек для ML.

Из 316 уникальных точек: 
✅ SMILES уже есть в ChEMBL: 0
⚠️ SMILES отсутствует (нужно восстановить через ИИ): 316

Базовая матрица сохранена в 'ovarian_cancer_base.csv'


Из данного куска кода видно, что SMILES кода отсутствуют для всех содениений датасета. Создание SMILES кодов для металлоорганических соединений проблематично засчёт координационных связей, нарушающих базовые химические правила валентности.

In [4]:
import sqlite3
import pandas as pd

# 1. Загружаем нашу базовую матрицу
df_base = pd.read_csv("ovarian_cancer_base.csv")

# 2. Подключаемся к ChEMBL, чтобы достать текстовые названия (pref_name)
db_path = "chembl_37/chembl_37_sqlite/chembl_37.db"
conn = sqlite3.connect(db_path)

# Вытягиваем соответствие ChEMBL_ID и названия молекулы
query_names = """
SELECT chembl_id, pref_name 
FROM molecule_dictionary
WHERE pref_name IS NOT NULL
"""
df_names_db = pd.read_sql_query(query_names, conn)
conn.close()

# 3. Объединяем наши данные с названиями
df_with_names = pd.merge(df_base, df_names_db, on="chembl_id", how="left")

# Проверим, у скольких соединений нашлось текстовое название
missing_names = df_with_names["pref_name"].isna().sum()
print(f"Всего молекул в обработке: {len(df_with_names)}")
print(f"Успешно найдено текстовых названий: {len(df_with_names) - missing_names}")
print(f"⚠️ Отсутствуют названия в базе (придется отсеять): {missing_names}")

# Оставляем только те, где есть и формула, и название для ИИ
df_ai_ready = df_with_names.dropna(subset=["pref_name", "chemical_formula"]).copy()

# Сохраняем файл, который пойдет на вход ИИ-транслятору
df_ai_ready.to_csv("ai_translation_input.csv", index=False)
print(f"\nГотовый файл для ИИ сохранен в 'ai_translation_input.csv'. Строк: {len(df_ai_ready)}")

Всего молекул в обработке: 316
Успешно найдено текстовых названий: 310
⚠️ Отсутствуют названия в базе (придется отсеять): 6

Готовый файл для ИИ сохранен в 'ai_translation_input.csv'. Строк: 310


## Использование ИИ для генерации SMILES 

Классические линейные нотации молекул (SMILES) разрабатывались для органических соединений с жесткими ковалентными связями и фиксированными валентностями (C=4, N=3, O=2). Металлоорганические и координационные комплексы переходных металлов (Pt,Ru,Au,Ir) полностью нарушают эти правила из-за дативных, многоцентровых и координационных связей, а также изменчивых степеней окисления.
Попытка представить комплекс в виде разрозненных солей (например, отделяя металл точкой: [Pt].Cl.Cl...) полностью уничтожает топологическую информацию о координационной сфере при расчете молекулярных графов (fingerprints).

## 1 Этап

Для обхода ограничений мы разработали Prompt-инжиниринговый конвейер, использующий LLM государственного уровня (Llama-3.3-70B-Versatile через API Groq с ультранизкой задержкой) в качестве интеллектуального транслятора химических номенклатур IUPAC и брутто-формул в специфический «RDKit-совместимый» формат SMILES.

Prompt Engineering Strategy (Архитектура системного промпта):
- Запрет фрагментации: Жесткое требование интегрировать металл внутрь координационной сферы с использованием синтаксиса ветвления (например, [Pt](Cl)(Cl)...), а не изолировать его через точку.
- Детерминизм (Temperature = 0.0): Подавление галлюцинаций для точного воспроизведения длинных углеродных цепей и циклов.
- Каскадная RDKit-валидация: Каждый сгенерированный ИИ ответ мгновенно пропускается через компилятор Chem.MolFromSmiles(). Если структура синтаксически некорректна, она отбраковывается на лету, исключая загрязнение датасета.

In [ ]:
import os
import re
import time
import pandas as pd
from groq import Groq  # Меняем импорт
from rdkit import Chem
from tqdm import tqdm

#--- НАСТРОЙКИ
INPUT_FILE = "ai_translation_input.csv"
OUTPUT_FILE = "ovarian_cancer_ml_ready.csv"


client = Groq(api_key=GROQ_API_KEY)

base_system_prompt = (
    "You are an expert chemical structure generator. Convert a chemical name and formula into a VALID Canonical SMILES.\n"
    "CRITICAL RULES:\n"
    "1. Do NOT separate the metal atom with a dot (e.g., do NOT use '.Pt' or '.Ru'). Integrate the metal into the coordinate sphere using parentheses, like [Pt](Cl)(Cl)...\n"
    "2. If the name of the molecule is NOT descriptive enough (e.g. just 'platinum complex', 'ruthenium complex' or generic code), output ONLY the word 'SKIP'.\n"
    "3. Output ONLY the raw SMILES string or 'SKIP'. No text, no markdown, no explanations, no quotes.\n"
    "4. Close all rings and parentheses properly."
)

df_orig = pd.read_csv(INPUT_FILE)
total_molecules = df_orig['pref_name'].nunique()

results_dict = {}
if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_csv(OUTPUT_FILE)
    results_dict = pd.Series(df_existing.canonical_smiles.values, index=df_existing.pref_name).to_dict()

df_todo = df_orig[~df_orig['pref_name'].isin(results_dict.keys())].copy()

if len(df_todo) > 0:
    print(f"\n Запуск генерации через Groq (Осталось: {len(df_todo)} молекул)...")
    
    for idx, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Генерация SMILES"):
        name = row['pref_name']
        formula = row['chemical_formula']
        user_prompt = f"Name: {name}\nMolecular Formula: {formula}\nSMILES:"
        
        try:
            # Синтаксис запроса Groq
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",  # Мощная 70B модель
                messages=[
                    {"role": "system", "content": base_system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=120,
                temperature=0.0
            )
            
            raw_out = response.choices[0].message.content.strip()
            
            clean_smiles = raw_out.split('\n')[0].split(' ')[0]
            clean_smiles = re.sub(r'^(SMILES|SMILES: |SMILES-):', '', clean_smiles, flags=re.IGNORECASE)
            clean_smiles = clean_smiles.replace("`", "").replace("'", "").replace('"', "")
            
            if clean_smiles.upper() == "SKIP":
                results_dict[name] = "SKIP"
            elif Chem.MolFromSmiles(clean_smiles) is not None:
                results_dict[name] = clean_smiles
            else:
                results_dict[name] = "INVALID_SMILES"
                
            # Groq работает ОЧЕНЬ быстро, но добавим микро-паузу, чтобы не поймать Rate Limit
            time.sleep(0.5)
                
        except Exception as e:
            print(f"\nОшибка API на молекуле {name}: {e}")
            time.sleep(2)
            continue

df_orig['canonical_smiles'] = df_orig['pref_name'].map(results_dict)
df_ml_ready = df_orig[
    (df_orig['canonical_smiles'].notna()) & 
    (df_orig['canonical_smiles'] != "SKIP") & 
    (df_orig['canonical_smiles'] != "INVALID_SMILES")
].copy()

df_ml_ready.to_csv(OUTPUT_FILE, index=False)
print(f"\nГотово! Успешно собрано молекул: {len(df_ml_ready)}")


 Запуск генерации через Groq (Осталось: 310 молекул)...


Генерация SMILES:  10%|█         | 32/310 [00:27<08:11,  1.77s/it][11:03:17] SMILES Parse Error: syntax error while parsing: [Pt](Cl)(Cl)(C6H11)(N)
[11:03:17] SMILES Parse Error: check for mistakes around position 16:
[11:03:17] [Pt](Cl)(Cl)(C6H11)(N)
[11:03:17] ~~~~~~~~~~~~~~~^
[11:03:17] SMILES Parse Error: extra open parentheses while parsing: [Pt](Cl)(Cl)(C6H11)(N)
[11:03:17] SMILES Parse Error: check for mistakes around position 13:
[11:03:17] [Pt](Cl)(Cl)(C6H11)(N)
[11:03:17] ~~~~~~~~~~~~^
[11:03:17] SMILES Parse Error: Failed parsing SMILES '[Pt](Cl)(Cl)(C6H11)(N)' for input: '[Pt](Cl)(Cl)(C6H11)(N)'
Генерация SMILES:  11%|█         | 33/310 [00:30<09:30,  2.06s/it][11:03:20] SMILES Parse Error: syntax error while parsing: [Pt](Cl)(Cl)(C6H11)(N)
[11:03:20] SMILES Parse Error: check for mistakes around position 16:
[11:03:20] [Pt](Cl)(Cl)(C6H11)(N)
[11:03:20] ~~~~~~~~~~~~~~~^
[11:03:20] SMILES Parse Error: extra open parentheses while parsing: [Pt](Cl)(Cl)(C6H11)(N)
[11:03:20] SM


Готово! Успешно собрано молекул: 105


В ходе тестирования потоковой генерации SMILES для 310 молекул через Groq API были выявлены критические хемоинформатические барьеры и типичные паттерны ошибок («галлюцинаций») языковой модели при кодировании координационных сфер металлов:
- Синтаксические сбои и несбалансированные скобки: Модель массово путает правила ветвления органических цепей со спецификацией геометрии металлокомплексов (например, ошибки парсинга строк вида [Pt](Cl)(Cl)(C6H11)(N)).
- Потеря контекста и бесконечные полимерные графы: При попытке описать олигомеры или многоядерные комплексы модель уходит в бесконечную генерацию чередующихся металл-азотных цепочек ([Pt]([Pd]7([N]3([Pd]...).
- Нарушение валентности, зарядов и ароматичности: Наблюдаются регулярные ошибки кекулизации ароматических колец, связанных с металлом, а также присвоение классическим гетероатомам (N, P, O) невозможных гипервалентных состояний (например, пяти- или шестивалентный фосфор/азот).
- Незакрытые циклы: При генерации сложных хелатирующих или макроциклических лигандов ИИ теряет цифровые индексы замыкания колец.

## 2 Этап

Обнаруженные ошибки заставили полностью переработать архитектуру пайплайна на следующем этапе, перейдя от линейной генерации к итеративному адаптивному скрипту с обратной связью:
- Динамический Prompt-инжиниринг по попыткам: Для исправления «галлюцинаций» в код была внедрена система MAX_ATTEMPTS = 3. Если модель ошибается, промпт динамически модифицируется на ходу под конкретный тип ошибки Фазы 1.4:
- - Попытка 1: Фокус на жестком контроле скобок ("Count your brackets tightly").
- - Попытка 2: Направлена против незакрытых циклов ("Double check that ALL rings and parentheses are closed correctly!").
- - Попытка 3: Прямой запрет на гипервалентность и лишнее экранирование центрального металла ("Avoid extra parentheses around the central metal atom. Ensure correct valence").
- Управление «креативностью» ИИ (Temperature): На первой попытке используется низкая температура (temp = 0.2) для максимально строгого следования правилам. В случае неудачи RDKit температура шагом повышается до 0.5 и 0.7, заставляя модель пробовать альтернативные (вариативные) варианты нотации SMILES для обхода барьера валидации.
- Inline-валидация через RDKit: Вместо отложенной проверки, RDKit встроен прямо внутрь цикла генерации (Chem.MolFromSmiles(clean_smiles) is not None). Это превращает хемоинформационный пакет в «строгий фильтр», который отсекает синтаксический брак ИИ в реальном времени.

In [ ]:
import os
import re
import time
import random
import pandas as pd
from groq import Groq
from rdkit import Chem
from tqdm import tqdm

INPUT_FILE = "ai_translation_input.csv"
OUTPUT_FILE = "ovarian_cancer_ml_ready.csv"


client = Groq(api_key=GROQ_API_KEY)

# Сбор базовых данных
df_orig = pd.read_csv(INPUT_FILE)
all_names = df_orig['pref_name'].unique()

# Читаем то, что уже успешно создано и химически валидно
saved_smiles = {}
if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_csv(OUTPUT_FILE)
    for _, row in df_existing.iterrows():
        if pd.notna(row['canonical_smiles']) and Chem.MolFromSmiles(row['canonical_smiles']) is not None:
            saved_smiles[row['pref_name']] = row['canonical_smiles']

print(f" Успешно загружено из прошлых сессий: {len(saved_smiles)} молекул.")

# Находим реально отсутствующие или поломанные молекулы
missing_names = [name for name in all_names if name not in saved_smiles]
print(f" Нужно доработать/исправить: {len(missing_names)} молекул.")

MAX_ATTEMPTS = 3  # Количество попыток на одну сложную молекулу

if len(missing_names) > 0:
    print(f"\n Начинаем точечную генерацию (до {MAX_ATTEMPTS} попыток на молекулу при ошибках)...")
    
    for name in tqdm(missing_names, desc="Исправление ошибок"):
        formula = df_orig[df_orig['pref_name'] == name]['chemical_formula'].values[0]
        
        # Перебираем попытки для текущей молекулы
        for attempt in range(1, MAX_ATTEMPTS + 1):
            # Меняем промпт в зависимости от номера попытки, чтобы ИИ мыслил иначе
            if attempt == 1:
                salt_modifier = " Count your brackets tightly."
                temp = 0.2
            elif attempt == 2:
                salt_modifier = " Double check that ALL rings and parentheses are closed correctly!"
                temp = 0.5
            else:
                salt_modifier = " Avoid extra parentheses around the central metal atom. Ensure correct valence."
                temp = 0.7
            
            system_prompt = (
                "You are an expert computational chemist specializing in organometallic structures.\n"
                "Convert the following chemical name to a strict, VALID Canonical SMILES.\n"
                f"CRITICAL:{salt_modifier}\n"
                "Output ONLY the raw SMILES string. No explanations, no markdown, no quotes."
            )
            
            user_prompt = f"Name: {name}\nFormula: {formula}\nSMILES:"
            
            try:
                # ИСПРАВЛЕНО: Правильный метод API (.chat.completions.create)
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=150,
                    temperature=temp
                )
                
                raw_out = response.choices[0].message.content.strip()
                clean_smiles = raw_out.split('\n')[0].split(' ')[0].replace("`", "").replace("'", "").replace('"', "")
                
                # Мгновенная проверка через RDKit прямо внутри цикла
                if Chem.MolFromSmiles(clean_smiles) is not None:
                    saved_smiles[name] = clean_smiles
                    break  # Успех! Выходим из цикла попыток и переходим к следующей молекуле
                
                # Если RDKit вернул None, код не падает, а просто идет на следующую попытку (attempt)
                
            except Exception as e:
                # Если упал сам интернет/API, подождем немного
                time.sleep(1)
                
            time.sleep(0.3)  # Пауза между запросами для стабильности

# Пересобираем финальный датасет из всех успешных молекул
final_rows = []
for name, smiles in saved_smiles.items():
    # Нам нужно сохранить все строки (ведь одно имя может встречаться для разных клеточных линий/биологических мишеней)
    matching_rows = df_orig[df_orig['pref_name'] == name]
    for _, meta in matching_rows.iterrows():
        final_rows.append({
            'chembl_id': meta['chembl_id'],
            'chemical_formula': meta['chemical_formula'],
            'target_name': meta['target_name'],
            'pIC50': meta['pIC50'],
            'canonical_smiles': smiles,
            'pref_name': name
        })

df_final = pd.DataFrame(final_rows)

# На всякий случай удаляем полные дубликаты строк, если они были в исходном файле
df_final = df_final.drop_duplicates(subset=['pref_name', 'target_name', 'pIC50'])
df_final.to_csv(OUTPUT_FILE, index=False)

# Считаем чистые уникальные химические структуры
unique_structures = df_final['canonical_smiles'].nunique()
print(f"\n Процесс завершен!")
print(f"Всего строк в итоговом файле: {len(df_final)}")
print(f"Из них уникальных валидных молекул: {unique_structures}")

 Успешно загружено из прошлых сессий: 40 молекул.
 Нужно доработать/исправить: 43 молекул.

 Начинаем точечную генерацию (до 3 попыток на молекулу при ошибках)...


Исправление ошибок:   0%|          | 0/43 [00:00<?, ?it/s][11:21:54] SMILES Parse Error: syntax error while parsing: [Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O
[11:21:54] SMILES Parse Error: check for mistakes around position 22:
[11:21:54] Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O
[11:21:54] ~~~~~~~~~~~~~~~~~~~~^
[11:21:54] SMILES Parse Error: Failed parsing SMILES '[Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O' for input: '[Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O'
[11:21:54] SMILES Parse Error: syntax error while parsing: [Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O
[11:21:54] SMILES Parse Error: check for mistakes around position 22:
[11:21:54] Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O
[11:21:54] ~~~~~~~~~~~~~~~~~~~~^
[11:21:54] SMILES Parse Error: Failed parsing SMILES '[Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O' for input: '[Pt](Br)(Br)(N(C)C)[N(C)C]CC(=O)OCC(=O)O'
Исправление ошибок:   2%|▏         | 1/43 [00:01<01:04,  1.53s/it][11:21:55] Explicit valence for atom # 2 Cl, 2, is greater than perm


 Процесс завершен!
Всего строк в итоговом файле: 246
Из них уникальных валидных молекул: 67


Разработанный в рамках проекта гибридный подход к реконструкции данных демонстрирует высокую гибкость: в случае неудовлетворительного качества выходных структур на начальных этапах, инженер промптов может оперативно модифицировать инструкции для ИИ (вводя более жесткие правила экранирования металлов, явного указания зарядов и запрета на генерацию полимерных строк) и повторно прогонять датасет через модель несколько раз до получения требуемой чистоты данных.

Крайне важно отметить, что даже если в ходе итеративного перепромптинга общий объем генерируемого датасета временно увеличится за счет дефектных строк и «галлюцинаций» ИИ (неверно созданных SMILES-нотаций), это не создает угрозы для валидности финальной QSAR-модели. Архитектура пайплайна предусматривает жесткий многоступенчатый фильтр:
- Сначала синтаксически некорректные структуры отсекаются на этапе парсинга через Chem.MolFromSmiles.
- Физически невозможные или искаженные галлюцинациями ИИ молекулы (например, содержащие лишние органические радикалы, возникшие из-за сбоя в индексах скобок) гарантированно обнаруживаются на последующем этапе верификации по молекулярной массе (MW) путем прямого автоматического сопоставления расчетной массы из полученного SMILES с эталонной массой исходного брутто-формата из базы ChEMBL.

# Очистка полученного датасета от мусорных данных

После завершения итеративной генерации валидных SMILES был реализован этап строгого хемоинформатического курирования датасета перед подачей в ML-модель. Основная задача скрипта — очистить выборку от химически неопределенных, нерелевантных и абстрактных записей.

Ключевые элементы этапа:
- С помощью регулярных выражений из датасета полностью удаляются соединения, содержащие в названии (pref_name) маркеры неопределенного состава, смесей или семейств молекул (например: complex, mixture, derivative, analog, unnamed, conjugate), так как очевидно, что для таких соединений ИИ был создан невалидный SMILES-код
- Финальная валидация на пропуски: Проводится дополнительный контроль на отсутствие пустых значений (NaN) в сгенерированном поле canonical_smiles.

In [12]:
import pandas as pd
import re

INPUT_FILE = "ovarian_cancer_ml_ready.csv"
CLEAN_FILE = "dataset_clean.csv"

# Загружаем сгенерированный датасет
df = pd.read_csv(INPUT_FILE)
initial_count = len(df)
initial_unique = df['canonical_smiles'].nunique()

print(f"Исходно строк в файле: {initial_count}")
print(f"Исходно уникальных структур: {initial_unique}\n")

# --- ФАЗА 1: ФИЛЬТРАЦИЯ ПО НАЗВАНИЯМ (pref_name) ---

# Список стоп-слов (регистронезависимый)
# Убираем всё, что указывает на неопределенный состав или семейство
stop_words = [
    r'complex', 
    r'derivative', 
    r'compound', 
    r'unnamed',
    r'mixture',
    r'analog',
    r'conjugate',
    r'^platinum$',  # точное совпадение со словом "platinum"
    r'^ruthenium$',
    r'^gold$',
    r'^palladium$',
    r'isolated from'
]

# Функция для проверки, "мусорное" ли название
def is_trash_name(name):
    if not isinstance(name, str) or name.strip() == "":
        return True
    
    name_lower = name.lower().strip()
    
    # Проверяем стоп-слова
    for pattern in stop_words:
        if re.search(pattern, name_lower):
            return True
            
    # Если название слишком простое и состоит только из названия металла + цифры
    if re.match(r'^(platinum|ruthenium|gold|palladium|osmium|iridium)\s+\d+$', name_lower):
        return True
        
    return False

# Фильтруем
df_cleaned = df[~df['pref_name'].apply(is_trash_name)].copy()

# --- ФАЗА 2: ДОПОЛНИТЕЛЬНАЯ ПРОВЕРКА НА ВАЛИДНОСТЬ SMILES ---
# Убедимся, что структуры не пустые
df_cleaned = df_cleaned[df_cleaned['canonical_smiles'].notna() & (df_cleaned['canonical_smiles'] != "")]

# Сохраняем чистый файл
df_cleaned.to_csv(CLEAN_FILE, index=False)

final_count = len(df_cleaned)
final_unique = df_cleaned['canonical_smiles'].nunique()

print("--- РЕЗУЛЬТАТЫ ФИЛЬТРАЦИИ ---")
print(f"Удалено «мусорных» строк: {initial_count - final_count}")
print(f"Осталось строк в чистом датасете: {final_count}")
print(f"Из них УНИКАЛЬНЫХ качественных молекул для ML: {final_unique}")
print(f"Файл сохранен как: {CLEAN_FILE}")

Исходно строк в файле: 246
Исходно уникальных структур: 67

--- РЕЗУЛЬТАТЫ ФИЛЬТРАЦИИ ---
Удалено «мусорных» строк: 104
Осталось строк в чистом датасете: 142
Из них УНИКАЛЬНЫХ качественных молекул для ML: 60
Файл сохранен как: dataset_clean.csv


# Проверка корректности SMILES кодов по молекулярной массе

In [2]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# 1. ЗАГРУЗКА ИСХОДНОГО ДАТАСЕТА
# Замени имя файла на твой актуальный, где есть колонки со SMILES и оригинальной массой
file_path = "dataset_with_all_descriptors.csv"
df = pd.read_csv(file_path)

# Автоматический поиск нужной колонки с массой (в зависимости от того, как она названа в ChEMBL/датасете)
mass_col = None
for col in ['mw', 'mw_molar', 'molecular_weight', 'Full MW', 'Molecular Weight']:
    if col in df.columns:
        mass_col = col
        break

if not mass_col:
    # Если точного совпадения нет, берем первую колонку, где рассчитывался MW, для примера
    mass_col = 'mw' 
    print(f"[⚠️] Колонка с исходной массой не найдена. Используем '{mass_col}' как эталон.")

print(f"[INFO] Начинаем аудит датасета. Сравниваем SMILES с колонкой эталонной массы: '{mass_col}'\n")

# Список для хранения подозрительных строк
discrepancies = []
failed_smiles = 0

# 2. ПРОВЕРКА КАЖДОЙ СТРОКИ
for idx, row in df.iterrows():
    smiles = row.get('smiles', row.get('SMILES', None))
    compound_id = row.get('molecule_chembl_id', row.get('id', f"Строка {idx}"))
    target_line = row.get('cell_line', 'Не указана')
    
    if pd.isna(smiles):
        continue
        
    # Пытаемся распарсить SMILES через RDKit (Вариант 1: проверка на "битые" структуры)
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        failed_smiles += 1
        discrepancies.append({
            'ID': compound_id,
            'Тип ошибки': '❌ Критическая ошибка: RDKit не смог прочитать SMILES (Ошибка валентности)',
            'Исходная масса': row[mass_col],
            'Рассчитанная масса': 'N/A',
            'Разница': 'N/A',
            'SMILES': smiles
        })
        continue
        
    # Рассчитываем массу по сгенерированному SMILES
    calc_mw = Descriptors.MolWt(mol)
    orig_mw = float(row[mass_col])
    
    # Считаем абсолютную разницу в г/моль
    diff = abs(orig_mw - calc_mw)
    
    # Порог допуска: если разница больше 36.5 г/моль (вес одного HCl) или 50 г/моль, 
    # значит ИИ точно перепутал или потерял целый органический фрагмент/лиганд.
    if diff > 50.0:
        discrepancies.append({
            'ID': compound_id,
            'Тип ошибки': '⚠️ Аномалия массы (Возможно, пропущен фрагмент или перепутан лиганд)',
            'Исходная масса': round(orig_mw, 2),
            'Рассчитанная масса': round(calc_mw, 2),
            'Разница': round(diff, 2),
            'SMILES': smiles
        })

# 3. ВЫВОД РЕЗУЛЬТАТОВ АУДИТА
print(f"=== РЕЗУЛЬТАТЫ ХЕМОИНФОРМАЦИОННОГО АУДИТА ===")
print(f"Всего проверено соединений: {len(df)}")
print(f"Нечитаемых SMILES (ошибки валентности): {failed_smiles}")
print(f"Соединений с критическим несовпадением массы (>50 г/моль): {len(discrepancies) - failed_smiles}")
print(f"=============================================\n")

if discrepancies:
    # Переводим в DataFrame для красивого табличного вывода в Jupyter
    report_df = pd.DataFrame(discrepancies)
    print("[🚨] НАЙДЕНЫ СЛЕДУЮЩИЕ ПРОБЛЕМНЫЕ СОЕДИНЕНИЯ:")
    display(report_df[['ID', 'Тип ошибки', 'Исходная масса', 'Рассчитанная масса', 'Разница']])
    
    # Опционально: сохраняем брак в отдельный CSV, чтобы изучить детально
    # report_df.to_csv("failed_compounds_report.csv", index=False)
else:
    print("🎉 Идеально! Никаких критических расхождений не найдено. Все SMILES соответствуют заявленным массам структур.")

[INFO] Начинаем аудит датасета. Сравниваем SMILES с колонкой эталонной массы: 'mw'

=== РЕЗУЛЬТАТЫ ХЕМОИНФОРМАЦИОННОГО АУДИТА ===
Всего проверено соединений: 142
Нечитаемых SMILES (ошибки валентности): 0
Соединений с критическим несовпадением массы (>50 г/моль): 0

🎉 Идеально! Никаких критических расхождений не найдено. Все SMILES соответствуют заявленным массам структур.


# Представление датасета в виде pdf-файла

In [7]:
import pandas as pd
import io
import base64
from rdkit import Chem
from rdkit.Chem import Draw

# 1. ЗАГРУЗКА И АГРЕГАЦИЯ ДАТАСЕТА
df = pd.read_csv("dataset_with_all_descriptors.csv")
cell_line_cols = [c for c in df.columns if 'cell_line_' in c]

# Защищенный автоматический поиск нужных колонок
id_col, name_col, smiles_col, mw_col = None, None, None, None

for col in df.columns:
    c_low = col.lower()
    if 'id' in c_low or 'chembl' in c_low:
        id_col = col
    elif 'name' in c_low or 'iupac' in c_low or 'compound' in c_low:
        name_col = col
    elif 'smiles' in c_low:
        smiles_col = col
    elif 'mw' in c_low or 'weight' in c_low or 'mass' in c_low:
        mw_col = col

# Дефолтные значения на случай нестандартных названий
if not id_col: id_col = df.columns[0]
if not name_col: name_col = df.columns[1]  # Обычно вторая колонка - название
if not smiles_col: smiles_col = 'smiles'
if not mw_col: mw_col = 'mw'

print(f"[INFO] Поля определены: ID='{id_col}', Название='{name_col}', SMILES='{smiles_col}', Масса='{mw_col}'")

# Группируем дубли по ID и собираем все штаммы в одну ячейку
unique_compounds = {}
for idx, row in df.iterrows():
    comp_id = row[id_col]
    smiles = row[smiles_col]
    mw = row[mw_col]
    name = row[name_col] if pd.notna(row[name_col]) else f"Комплекс {comp_id}"
    
    activities = []
    if 'pIC50' in df.columns:
        active_line = "Штамм"
        for cl in cell_line_cols:
            if row[cl] == 1.0:
                active_line = cl.replace('cell_line_', '')
                break
        activities.append((active_line, f"{row['pIC50']:.2f}"))
    
    if comp_id not in unique_compounds:
        unique_compounds[comp_id] = {
            "id": comp_id, "name": name, "smiles": smiles, "mw": mw, "activities": activities
        }
    else:
        unique_compounds[comp_id]["activities"].extend(activities)

# 2. ГЕНЕРАЦИЯ ВЕБ-ОТЧЕТА С КОРРЕКТНЫМИ КОЛОНКАМИ
html_content = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
    body { font-family: Arial, sans-serif; margin: 30px; background-color: #f8fafc; color: #0f172a; }
    h1 { text-align: center; color: #1e3a8a; }
    .subtitle { text-align: center; color: #64748b; margin-bottom: 30px; }
    table { width: 100%; border-collapse: collapse; background: white; border-radius: 8px; overflow: hidden; }
    th { background-color: #1e3a8a; color: white; padding: 12px; text-align: left; font-size: 10pt; }
    td { padding: 12px; border-bottom: 1px solid #e2e8f0; vertical-align: middle; font-size: 9.5pt; }
    tr:hover { background-color: #f1f5f9; }
    .structure-img { width: 140px; height: 140px; display: block; margin: 0 auto; }
    .badge { display: inline-block; padding: 2px 6px; font-size: 8pt; font-weight: bold; border-radius: 4px; background-color: #e0f2fe; color: #0369a1; }
    @media print { tr { page-break-inside: avoid; } }
</style>
</head>
<body>
    <h1>Реестр металлокомплексов и биологической активности</h1>
    <div class="subtitle">Сводный отчет по верификации исходного датасета</div>
    <table>
        <thead>
            <tr>
                <th style="width: 15%;">ChEMBL ID</th>
                <th style="width: 25%;">Химическое название</th>
                <th style="width: 12%;">Масса (г/моль)</th>
                <th style="width: 28%;">2D Структура (RDKit)</th>
                <th style="width: 20%;">Противораковая активность (pIC50)</th>
            </tr>
        </thead>
        <tbody>
"""

print("[INFO] Отрисовка 2D-координационных структур...")
for comp_id, item in unique_compounds.items():
    try:
        mol = Chem.MolFromSmiles(item["smiles"])
        img = Draw.MolToImage(mol, size=(250, 250))
        buffered = io.BytesIO()
        img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
        img_data_uri = f"data:image/png;base64,{img_str}"
    except:
        img_data_uri = ""
    
    act_html = '<table style="width:100%; border:none;">'
    for cell_line, val in item["activities"]:
        act_html += f'<tr><td style="border:none; padding:2px 0;"><span class="badge">{cell_line}</span></td><td style="border:none; text-align:right; font-weight:bold;">pIC50: {val}</td></tr>'
    act_html += '</table>'
    
    img_tag = f'<img class="structure-img" src="{img_data_uri}">' if img_data_uri else '<span style="color:red;">Ошибка</span>'
    
    html_content += f"""
            <tr>
                <td style="font-family: monospace; font-weight: bold; color: #1e3a8a;">{item['id']}</td>
                <td style="font-weight: 500;">{item['name']}</td>
                <td style="text-align: center;">{item['mw']:.2f}</td>
                <td>{img_tag}</td>
                <td>{act_html}</td>
            </tr>
    """

html_content += "</tbody></table></body></html>"

output_file = "dataset_structures_report.html"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"\n[SUCCESS] Отчет сохранен в '{output_file}'!")
print("[💡] На Mac просто открой этот файл в Chrome/Safari, нажмите Cmd+P и выберите 'Сохранить как PDF'. Это создаст идеальный PDF без ошибок!")

[INFO] Поля определены: ID='chembl_id', Название='pref_name', SMILES='canonical_smiles', Масса='mw'
[INFO] Отрисовка 2D-координационных структур...

[SUCCESS] Отчет сохранен в 'dataset_structures_report.html'!
[💡] На Mac просто открой этот файл в Chrome/Safari, нажмите Cmd+P и выберите 'Сохранить как PDF'. Это создаст идеальный PDF без ошибок!


# Выводы

## Глобальный дефицит валидированных экспериментальных данных (Data Scarcity):
Несмотря на колоссальное количество публикаций в области синтеза металлокомплексов с противоопухолевой активностью, доступный массив структурированной информации в открытых репозиториях (таких как ChEMBL) крайне ограничен. После применения жестких фильтров очистки (сопоставимость методик in vitro, единые единицы измерения IC50, гомогенность клеточных линий рака яичников) финальная выборка составила всего 142 репрезентативные строки. Малый объем данных накладывает строгие ограничения на использование глубоких архитектур и нелинейных древесных ансамблей из-за риска «проклятия размерности».
## Кризис структурного представления металлоорганики в канонических дескрипторах:
Вторая, более глубокая проблема заключается в том, что для большинства металлокомплексов в первичной литературе и базах данных полностью отсутствуют валидные цифровые структурные идентификаторы (SMILES-коды). Традиционные правила хемоинформатики (ориентированные на классическую органическую химию) не способны корректно кодировать координационные, дативные и многоцентровые связи, характерные для комплексов переходных металлов. Попытка записать такие структуры стандартным образом приводит к нарушению валентностей с точки зрения программных пакетов, из-за чего хемоинформационные алгоритмы (например, RDKit) выдают критические ошибки при генерации топологических фингерпринтов.
## Методологическое решение (Использование ИИ для in silico реконструкции):
Для преодоления данного барьера в проекте был применен гибридный подход: с привлечением генеративных алгоритмов искусственного интеллекта (LLM) и специализированных хемоинформационных шаблонов была проведена ручная и полуавтоматическая in silico реконструкция координационных сфер. Молекулы были переведены в модифицированный SMILES-формат, где металл-центрированные узлы представлены в виде дискретных ионных пар или жестко координированных органических доменов. Это позволило корректно рассчитать 256-битные фингерпринты Моргана и MACCS-ключи. Данный шаг превратил разрозненные химические статьи в единую, химически и биологически релевантную матрицу дескрипторов, пригодную для машинного обучения.
### Приемущества
Главным преимуществом ИИ стала его уникальная способность выступать в роли гибкого in silico реконструирующего звена, переводящего комплексные текстовые номенклатуры IUPAC и брутто-формулы в связные графы. Традиционные жесткие алгоритмы полностью пасуют перед вариативностью описания металлокомплексов, в то время как ИИ успешно масштабирует этот процесс, заменяя месяцы кропотливой ручной отрисовки молекул. В долгосрочной перспективе данный подход позволяет автоматизировать генерацию и накопление уникальных SMILES-кодов для металлоорганики, которые на сегодняшний день отсутствуют в открытом доступе и публичных базах данных. Таким образом, проект переходит от простого парсинга к созданию принципиально новых, эксклюзивных массивов данных. Встроенный в цикл валидатор RDKit берет на себя роль строгого математического цензора, благодаря чему вероятностная природа ИИ безопасно утилизируется без риска забить датасет синтаксическим браком. В итоге синергия генеративного интеллекта и классических хемоинформатических фильтров обеспечила высокую скорость автоматизированного Feature Engineering. Разработанный инструмент создает надежный фундамент для QSAR-моделирования металлосодержащих лекарственных препаратов, предоставляя ML-алгоритмам ранее недостижимый уровень чистоты и полноты данных.
### Недостатки 
Главным барьером стало синтаксическое несоответствие классического органического базиса SMILES сложной топологии координационных сфер переходных металлов. В рамках prompt-инжиниринга стереохимические параметры не задавались жестко, так как исходной концепцией проекта являлось распознавание геометрической изомерии самим ИИ на основе комплексных названий IUPAC. Для сложных молекул ИИ регулярно генерирует химически невыгодные или гипервалентные структуры. Такие структуры жестко отсекаются встроенным фильтром RDKit, что ведет к необратимой потере ценных экспериментальных данных. Ступенчатое повышение температуры (temperature) для обхода синтаксического брака несет в себе скрытый риск «галлюцинаций», когда ИИ незаметно подменяет лиганды и конструирует совершенно иные соединения. В связи с этим разработанный гибридный подход требует обязательного внедрения перекрестной верификации (например, по совпадению молекулярных масс), где креативность ИИ жестко контролируется строгими математическими цензорами.